In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import copy
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler


train = pd.read_excel("BNT-Energy2f.xlsx")
X=train.iloc[:,:-1]
Y=train.iloc[:,-1]

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=42)


scalerX = StandardScaler().fit(X_train)
scalery = StandardScaler().fit(y_train.values.reshape(-1,1))
X_train1 = scalerX.transform(X_train)
y_train1 = scalery.transform(y_train.values.reshape(-1,1))
X_test1 = scalerX.transform(X_test)
y_test1 = scalery.transform(y_test.values.reshape(-1,1))
#y_new_inverse = scalery.inverse_transform(y_test)
X_train2=copy.deepcopy(X_train1)
y_train2=copy.deepcopy(y_train1)
train.corr()
sns.heatmap(train.corr(), cmap="YlGnBu", annot=False)

import matplotlib.pyplot as plt
import seaborn as sns

corr = train.corr() #your dataframe

#figsize=(6, 6) control width and height
# dpi = 600, I
#plt.figure(figsize=(12, 12),
#        dpi = 600)

# parameter annot_kws={"size": 8} control corr values font size
#sns.heatmap(corr, cmap="Blues", annot=True, annot_kws={"size": 3})

plt.tick_params(axis = 'x', labelsize = 6) # x font label size
plt.tick_params(axis = 'y', labelsize = 6) # y font label size.....remove highly correlated features away then run

In [ ]:
from sklearn.ensemble import RandomForestRegressor
rf=RandomForestRegressor(random_state=42)
from pprint import pprint
import numpy as np
from sklearn.metrics import mean_absolute_error,r2_score,mean_squared_error

pprint(rf.get_params())

from sklearn.model_selection import RandomizedSearchCV

# Number of trees in random forest
n_estimators = [int(x) for x in np.linspace(start = 200, stop = 2000, num = 10)]
# Number of features to consider at every split
max_features = ['log2', 'sqrt']
# Maximum number of levels in tree
max_depth = [int(x) for x in np.linspace(2, 100, num = 50)]
max_depth.append(None)
# Minimum number of samples required to split a node
min_samples_split = [5, 10, 15]
# Minimum number of samples required at each leaf node
min_samples_leaf = [3, 4, 5, 8]
# Method of selecting samples for training each tree
bootstrap = [True, False]
# Create the random grid
random_grid = {'n_estimators': n_estimators,
               'max_features': max_features,
               'max_depth': max_depth,
               'min_samples_split': min_samples_split,
               'min_samples_leaf': min_samples_leaf,
               'bootstrap': bootstrap}
pprint(random_grid)

# Use the random grid to search for best hyperparameters
# First create the base model to tune
rf = RandomForestRegressor()
# Random search of parameters, using 3 fold cross validation,
# search across 100 different combinations, and use all available cores
rf_random = RandomizedSearchCV(estimator = rf, param_distributions = random_grid, n_iter = 100, cv = 5, verbose=3, random_state=42, n_jobs = 1)
# Fit the random search model
rf_random.fit(X_train1, np.ravel(y_train1))
rf_random.best_params_

In [ ]:
from sklearn.metrics import mean_squared_error
bootstrap_means=[]
bootstrap_means2=[]

N=500

model_rf=RandomForestRegressor(n_estimators=800,max_depth=76,min_samples_split=5,min_samples_leaf=3,max_features='sqrt',bootstrap=True)

for i in range(0,N):
  sample_index=np.random.choice(range(0,len(y_train1)),len(y_train1))

  X_samples=X_train1[sample_index]
  y_samples=y_train1[sample_index]
  print(i)

  model_rf.fit(X_samples, y_samples)
  y_prediction=model_rf.predict(X_test1)
  y_train2_prediction=model_rf.predict(X_train2)

  bootstrap_means.append(scalery.inverse_transform(y_prediction.reshape(-1,1)))
  bootstrap_means2.append(scalery.inverse_transform(y_train2_prediction.reshape(-1,1)))

bootstrap_means_array=np.array(bootstrap_means)
bootstrap_means2_array=np.array(bootstrap_means2)

print(r2_score(scalery.inverse_transform(y_test1),bootstrap_means_array.mean(0)))
#print(mean_squared_error(scalery.inverse_transform(y_test1),bootstrap_means_array.mean(0)))
print(np.sqrt(mean_squared_error(scalery.inverse_transform(y_test1),bootstrap_means_array.mean(0))))

print(r2_score(scalery.inverse_transform(y_train2),bootstrap_means2_array.mean(0)))
#print(mean_squared_error(scalery.inverse_transform(y_train2),bootstrap_means2_array.mean(0)))
print(np.sqrt(mean_squared_error(scalery.inverse_transform(y_train2),bootstrap_means2_array.mean(0))))

# Define file names for training and testing data
train_file = 'train_data_output_final.xlsx'
test_file = 'test_data_output_final.xlsx'

# Save the training data to an Excel file
train_data_output_final = pd.concat([X_train, y_train], axis=1)
train_data_output_final.to_excel(train_file, index=False)

# Save the testing data to an Excel file
test_data_output_final = pd.concat([X_test, y_test], axis=1)
test_data_output_final.to_excel(test_file, index=False)

# Add predictions to the 'test' DataFrame
test_data_output_final['Predicted_Wrec'] = bootstrap_means_array.mean(0)
test_data_output_final['Error'] = bootstrap_means_array.std(0)

# Add predictions to the 'train' DataFrame
train_data_output_final['Predicted_Wrec'] = bootstrap_means2_array.mean(0)
train_data_output_final['Error'] = bootstrap_means2_array.std(0)

# Save the updated DataFrame with predictions to a new file
output_file = 'test_predicted_Wrec.xlsx.'  # Update with the desired output file path
test_data_output_final.to_excel('test_predicted_Wrec-newIE2.xlsx', index=False)

# Save the updated DataFrame with predictions to a new file
output_file = 'train_predicted_Wrec.xlsx.'  # Update with the desired output file path
train_data_output_final.to_excel('train_predicted_Wrec-newIE2.xlsx', index=False)

In [ ]:
model_RF=RandomForestRegressor(n_estimators=800,max_depth=140,min_samples_split=2,min_samples_leaf=1,max_features='sqrt',bootstrap=False)

model_RF.fit(X_train1, y_train1)

dictionary={}
for col,score in zip(train.columns,model_RF.feature_importances_):
    print(col,score)
    dictionary[col]=score


#print(model_RF.feature_importances_)
keys = list(dictionary.keys())
values = list(dictionary.values())
sorted_value_index = np.argsort(values)
sorted_dict = {keys[i]: values[i] for i in sorted_value_index}
sorted_dict